# AgentRx — Test & Verification Notebook

**AgentRx** is a specialist swarm that diagnoses and patches *other* broken AI agents.
Today's target is **Meridian** — the Day-1 Exercise-04 coordinator agent, whose defects were
planted by Anthropic, not by us.

This notebook does three things:
1. Runs the **offline test suite** (52 tests, no API calls, no cost)
2. Verifies the **sandbox** hands the swarm the genuinely broken agent
3. Optionally replays or runs a **real diagnosis**

> Kernel: **Basecamp Live (.venv)** or **Python (Basecamp 3.13)**.
> Every cell below is independent — no hidden execution-order state.

In [ ]:
# Point Python at the agentrx package and use an ISOLATED sandbox so this
# notebook can never disturb a `record` run happening in another terminal.
import os, sys
from pathlib import Path


def _find_agentrx():
    """Locate agentrx/ relative to wherever this notebook was opened.

    This used to be a hardcoded absolute path, which meant the notebook ran on
    exactly one machine - in a public repo whose purpose is being run by other
    people."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "agentrx.py").exists():
            return candidate
        if (candidate / "agentrx" / "agentrx.py").exists():
            return candidate / "agentrx"
    raise SystemExit(
        "could not find agentrx.py from " + str(here) +
        " - open this notebook from inside the agentrx/ directory."
    )


AGENTRX = _find_agentrx()
os.environ["AGENTRX_SANDBOX"] = "sandbox_notebook"
os.environ["PYTHONUTF8"] = "1"
sys.path.insert(0, str(AGENTRX))
os.chdir(AGENTRX)

print("agentrx :", AGENTRX)
print("cwd     :", os.getcwd())
print("python  :", sys.version.split()[0])
print("sandbox :", os.environ["AGENTRX_SANDBOX"])

## 1 · Offline test suite

Parsing, patching, patch-verification, event bus, and the architectural claims.
No network, no billing.

In [ ]:
import tests
result = tests.run(verbosity=2)
print()
print(f"ran={result.testsRun}  failures={len(result.failures)}  errors={len(result.errors)}")
print("PASS" if result.wasSuccessful() else "FAIL")

## 2 · Sandbox verification

The swarm must receive the *pristine broken* agent, and the user's own
already-fixed exercise files must be left strictly alone.

In [ ]:
import sandbox, bridge

sandbox.reset()
prompt = Path(sandbox.SANDBOX_DIR, "system-prompt-coordinator.txt").read_text(encoding="utf-8")
real   = Path(sandbox.MERIDIAN_SRC, "system-prompt-coordinator.txt").read_text(encoding="utf-8")

print("planted defect present in sandbox   :", "exactly ONE specialist" in prompt)
print("user's real file left untouched     :", "exactly ONE specialist" not in real)
print("tool count (the context defect)     :", bridge.tool_manifest_stats()["tool_count"], "(17 = bloated baseline)")
print("vague/duplicate tools              :",
      [t for t in bridge.tool_manifest_stats()["tool_names"]
       if t in {"helper","process","get_data","tool_3_v2","validate"}])

## 3 · Patch + verification

A patch that *claims* success without landing on disk is precisely the defect
AgentRx exists to catch — so the patcher re-reads from disk to verify.

In [ ]:
import patcher

before = bridge.tool_manifest_stats()["tool_count"]
report = patcher.apply_patch({
    "system_prompt_find": "Each ticket is owned by exactly ONE specialist.",
    "system_prompt_replace": "Spawn every specialist the ticket actually requires.",
    "remove_tools": ["helper", "process", "get_data", "tool_3_v2", "validate"],
    "rationale": "notebook demo of patch + verify",
})
after = bridge.tool_manifest_stats()["tool_count"]

print("prompt changed :", report["prompt_changed"])
print("tools removed  :", report["tools_removed"])
print("tool count     :", before, "->", after)
print("VERIFIED       :", report["verified"], "(re-read from disk, not asserted)")
print("errors         :", report["errors"] or "none")

In [ ]:
# Negative control: a paraphrased (non-verbatim) find must FAIL loudly,
# never silently pretend to have patched.
sandbox.reset(verbose=False)
bad = patcher.apply_patch({
    "system_prompt_find": "this sentence is not in the prompt",
    "system_prompt_replace": "whatever",
    "remove_tools": [], "rationale": "negative control",
})
print("verified :", bad["verified"], "(expected False)")
print("errors   :", bad["errors"])

## 4 · The swarm roster

Three specialists, each blind to the others' lane. None is told a taxonomy exists —
the Attending discovers the defect families from what independently comes back.

In [ ]:
import specialists, attending

for s in specialists.SPECIALISTS:
    print(f"{s['name']:26s} {s['model']:20s} lane={s['key']}")
print()
print("Attending model :", "claude-opus-4-7")
print("stream timeout  :", attending.STREAM_TIMEOUT_SECONDS, "s")
print("taxonomy leaked to Attending :",
      "three defect families" in attending.COORDINATOR_SYSTEM.lower())

## 5 · Replay the recorded diagnosis  *(zero network calls)*

This is the on-stage command. It reads `artifacts/full_run.json` and prints the
beats with no API calls at all — a live pitch can't be broken by wifi.

In [ ]:
import agentrx, json

report = agentrx._read_artifact("full_run.json")
if report is None:
    print("No recorded run yet. Run this in a terminal first (several minutes, real API calls):")
    print("    python agentrx.py --trials 3 --iterations 2 --holdout record")
else:
    b, ft = report["baseline"]["eval"], report["final_train"]
    print(f"baseline train : {b['resolved_trials']}/{b['total_trials']} resolved  "
          f"({agentrx._fmt_cost(b.get('cost_total'))})")
    print(f"final train    : {ft['resolved_trials']}/{ft['total_trials']} resolved")
    # Through the shared summary. Printing the dict rendered the harness's
    # entire captured stdout as a Python repr - thousands of characters, in a
    # committed demo-facing file, onto a projector.
    for i, line in enumerate(agentrx._holdout_summary(report["holdout"])):
        print(f"holdout        : {line}" if i == 0 else f"                 {line}")
    tb, ta = report["tool_stats_before"], report["tool_stats_after"]
    print(f"tool count     : {tb['tool_count']} -> {ta['tool_count']}")
    for it in report["iterations"]:
        print(f"  iteration {it['iteration']}: {it.get('outcome','patched')}")

## 6 · Optional — run a REAL diagnosis  *(costs money, several minutes)*

Uncomment to dispatch the live swarm: 3 specialists in parallel + an Opus Attending,
then patch, re-evaluate, and holdout-gate. Watch the sidebar at
[http://localhost:8765](http://localhost:8765) while it runs (`python server.py`).

In [ ]:
# --- uncomment to run for real ---
# class Args:
#     trials, iterations, holdout, model, live = 3, 2, True, None, False
# report = agentrx.cmd_record(Args())
# print(report["final_train"], report["holdout"])

## 7 · Self-diagnosis — AgentRx audited against its own taxonomy

Split deliberately into a **measured** half and an **unverified** half. A self-critique
section is the easiest place in the whole build to commit the exact sin it warns about
(confident unverified claims), so the two never share a bucket.

In [ ]:
class A: pass
try:
    agentrx.cmd_selfcheck(A())
except SystemExit as e:
    print(e)